In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!tar -xf '/content/drive/MyDrive/laryngeal dataset.tar'

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import random
import seaborn as sns
import re

In [ ]:
def load_data_from_folder(data_folder):
    images = []
    labels = []
    for class_folder in os.listdir(data_folder):
        class_path = os.path.join(data_folder, class_folder)
        if os.path.isdir(class_path):
            for image_file in os.listdir(class_path):
                image_path = os.path.join(class_path, image_file)
                image = cv2.imread(image_path)
                if image is not None:
                    images.append(image)
                    labels.append(class_folder)
    return images, labels

data_folder1 = "/content/laryngeal dataset/FOLD 1"
data_folder2 = "/content/laryngeal dataset/FOLD 2"
data_folder3 = "/content/laryngeal dataset/FOLD 3"

images, labels = load_data_from_folder(data_folder1)
images1, labels1 = load_data_from_folder(data_folder2)
images2, labels2 = load_data_from_folder(data_folder3)

# Combine the images and labels from the three folds
images = images + images1 + images2
labels = labels + labels1 + labels2

In [ ]:
#checking number of images
len(images)

In [ ]:
# Display a random subset of images with their labels in subplots
num_images_to_display = 5
random_indices = random.sample(range(len(images)), num_images_to_display)

fig, axes = plt.subplots(1, num_images_to_display, figsize=(15, 3))

for i, idx in enumerate(random_indices):
    axes[i].imshow(cv2.cvtColor(images[idx], cv2.COLOR_BGR2RGB))
    axes[i].set_title("Label: " + str(labels[idx]))
    axes[i].axis("off")

plt.show()

In [ ]:
def preprocess_images(images, target_size=(100, 100)):
    preprocessed_images = []
    for image in images:
        resized_image = cv2.resize(image, target_size)
        normalized_image = resized_image.astype(np.float32) / 255.0
        preprocessed_images.append(normalized_image)
    return preprocessed_images

# Preprocess the images
images = preprocess_images(images)

In [ ]:
# Convert the data to numpy arrays
X = np.array(images)
y = np.array(labels)

In [ ]:

#print shape of X to see image size
X.shape

In [ ]:
#train test spliting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Flatten the images to 1D arrays (as SVM expects 1D input)
X_train_flat = np.array([image.flatten() for image in X_train])
X_test_flat = np.array([image.flatten() for image in X_test])

# Initialize and train the SVM
svm_classifier = SVC(probability=True)
svm_classifier.fit(X_train_flat, y_train)

In [ ]:
# Predict labels for the test set
y_pred = svm_classifier.predict(X_test_flat)

# Calculate accuracy on the test set
accuracy_test = accuracy_score(y_test, y_pred)
print("Testing Accuracy:", accuracy_test)

# Predict labels for the training set
y_pred_train = svm_classifier.predict(X_train_flat)

# Calculate accuracy on the training set
accuracy_train = accuracy_score(y_train, y_pred_train)
print("Training Accuracy:", accuracy_train)

In [ ]:
# Generate and print the classification report
class_names = np.unique(y)
classification_rep = classification_report(y_test, y_pred, target_names=class_names)
print("Classification Report:")
print(classification_rep)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
import pandas as pd

In [ ]:
Classes = class_names

In [ ]:
def results(model_name, y_pred, y_test, y_train,y_pred_train):
    target_names = ["{}".format(Classes[i]) for i in range(len(Classes))] # Define target names for classification report
    accuracy = round(accuracy_score(y_pred, y_test)*100,4)
    train_accuracy = round(accuracy_score(y_pred_train, y_train)*100,4)

    precision = round(precision_score(y_pred, y_test, average='macro')*100,4)
    recall = round(recall_score(y_pred, y_test, average='macro')*100,4)
    f1_scr = round(f1_score(y_pred, y_test, average='macro')*100,4)


    print("\nTraining Accuracy: {}%".format(train_accuracy))
    print("Testing Accuracy: {}%".format(accuracy))

    print("Precision: {}%".format(precision))
    print("Recall: {}%".format(recall))
    print("F1-Score: {}%".format(f1_scr))
    print()
    print("Classification Report:")
    print(classification_report(y_pred, y_test, target_names=target_names))
    print()
    print("Confusion Matrix:")
    fig, ax = plt.subplots(figsize=(7,5))
    ConfusionMatrixDisplay.from_predictions(y_pred, y_test,
                                            ax=ax,
                                            display_labels=target_names,
                                            xticks_rotation='vertical')
    plt.show()

    return {
        'Model':model_name,
        'Training Accuracy': train_accuracy,
        'Testing Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1_scr
    }

In [ ]:
def GetModelResultsInDataFrame(res):
  # Convert the dictionary to a DataFrame
  return pd.DataFrame.from_dict([res]).set_index('Model')

In [ ]:
svm_res = GetModelResultsInDataFrame(results("SVM", y_pred, y_test, y_train, y_pred_train))
svm_res

In [ ]:
# Initialize parameters for bootstrapping
num_bootstrap_samples = 10  # Number of bootstrap samples
ensemble_accuracies = []

# Perform bootstrapping
for _ in range(num_bootstrap_samples):
    # Create a bootstrap sample
    bootstrap_indices = np.random.choice(len(X_test_flat), len(X_test_flat), replace=True)
    bootstrap_X = X_test_flat[bootstrap_indices]
    bootstrap_y = y_test[bootstrap_indices]

    # Predict class labels using the bagging ensemble
    bootstrap_y_pred = svm_classifier.predict(bootstrap_X)

    # Calculate accuracy for the bootstrap sample
    bootstrap_accuracy = accuracy_score(bootstrap_y, bootstrap_y_pred)
    ensemble_accuracies.append(bootstrap_accuracy)

# Calculate mean and confidence interval for ensemble accuracy
mean_accuracy = np.mean(ensemble_accuracies)
lower_bound = np.percentile(ensemble_accuracies, 2.5)
upper_bound = np.percentile(ensemble_accuracies, 97.5)

# Print uncertainty quantification results
print(f"Ensemble Mean Accuracy: {mean_accuracy:.4f}")
print(f"Confidence Interval: {lower_bound:.4f} - {upper_bound:.4f}")

In [ ]:
y_prob = svm_classifier.predict_proba(X_test_flat)  # Probability estimates for each class

# Assuming you want to visualize uncertainty by plotting probability distributions
plt.figure(figsize=(12, 6))
for i, class_name in enumerate(class_names):
    class_probs = y_prob[:, i]
    plt.subplot(2, 5, i + 1)
    plt.hist(class_probs, bins=20, alpha=0.7, color='blue')
    plt.title(f'Class {class_name} Probabilities')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots to display images
num_classes = len(class_names)
num_images_per_class = 4  # You can adjust this as per your preference
fig, axes = plt.subplots(num_classes, num_images_per_class, figsize=(12, 12))

for i, class_name in enumerate(class_names):
    # Get indices of images for the current class
    class_indices = np.where(y_test == class_name)[0]

    # Randomly select num_images_per_class correct and incorrect predictions
    np.random.shuffle(class_indices)
    selected_correct_indices = class_indices[y_test[class_indices] == y_pred[class_indices]][:num_images_per_class // 2]
    selected_incorrect_indices = class_indices[y_test[class_indices] != y_pred[class_indices]][:num_images_per_class // 2]

    # Plot correct predictions
    for j, correct_idx in enumerate(selected_correct_indices):
        ax = axes[i, j * 2]
        ax.imshow(X_test[correct_idx], cmap='gray')
        ax.axis('off')
        ax.set_title(f"Correct: {class_name}")

    # Plot incorrect predictions
    for j, incorrect_idx in enumerate(selected_incorrect_indices):
        ax = axes[i, j * 2 + 1]
        ax.imshow(X_test[incorrect_idx], cmap='gray')
        ax.axis('off')
        predicted_label = y_pred[incorrect_idx]
        ax.set_title(f"Predicted: {predicted_label}, Actual: {class_name}")

# Adjust spacing between subplots
plt.tight_layout()
plt.show()


In [ ]:
# Define the hyperparameters to tune
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Initialize the SVM classifier
svm_classifier = SVC(probability=True)

# Perform grid search with cross-validation
grid_search = GridSearchCV(estimator=svm_classifier, param_grid=param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train_flat, y_train)

# Get the best hyperparameters from the grid search
best_params = grid_search.best_params_
print("Best Hyperparameters:", best_params)

# Train the SVM with the best hyperparameters on the full training data
best_svm_classifier = SVC(**best_params, probability=True)
best_svm_classifier.fit(X_train_flat, y_train)

# Predict labels for the test set
y_pred = best_svm_classifier.predict(X_test_flat)

# Calculate accuracy on the test set
accuracy_test = accuracy_score(y_test, y_pred)
print("Testing Accuracy:", accuracy_test)

# Predict labels for the training set
y_pred_train = best_svm_classifier.predict(X_train_flat)

# Calculate accuracy on the training set
accuracy_train = accuracy_score(y_train, y_pred_train)
print("Training Accuracy:", accuracy_train)

In [ ]:
# Generate and print the classification report
classification_rep = classification_report(y_test, y_pred, target_names=class_names)
print("Classification Report:")
print(classification_rep)

In [ ]:
svm_res1 = GetModelResultsInDataFrame(results("Tuned-SVM", y_pred, y_test, y_train, y_pred_train))
svm_res1

In [ ]:
# Initialize parameters for bootstrapping
num_bootstrap_samples = 10  # Number of bootstrap samples
ensemble_accuracies = []

# Perform bootstrapping
for _ in range(num_bootstrap_samples):
    # Create a bootstrap sample
    bootstrap_indices = np.random.choice(len(X_test_flat), len(X_test_flat), replace=True)
    bootstrap_X = X_test_flat[bootstrap_indices]
    bootstrap_y = y_test[bootstrap_indices]

    # Predict class labels using the bagging ensemble
    bootstrap_y_pred = best_svm_classifier.predict(bootstrap_X)

    # Calculate accuracy for the bootstrap sample
    bootstrap_accuracy = accuracy_score(bootstrap_y, bootstrap_y_pred)
    ensemble_accuracies.append(bootstrap_accuracy)

# Calculate mean and confidence interval for ensemble accuracy
mean_accuracy = np.mean(ensemble_accuracies)
lower_bound = np.percentile(ensemble_accuracies, 2.5)
upper_bound = np.percentile(ensemble_accuracies, 97.5)

# Print uncertainty quantification results
print(f"Ensemble Mean Accuracy: {mean_accuracy:.4f}")
print(f"Confidence Interval: {lower_bound:.4f} - {upper_bound:.4f}")

In [ ]:
y_prob = best_svm_classifier.predict_proba(X_test_flat)  # Probability estimates for each class

# Assuming you want to visualize uncertainty by plotting probability distributions
plt.figure(figsize=(12, 6))
for i, class_name in enumerate(class_names):
    class_probs = y_prob[:, i]
    plt.subplot(2, 5, i + 1)
    plt.hist(class_probs, bins=20, alpha=0.7, color='blue')
    plt.title(f'Class {class_name} Probabilities')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots to display images
num_classes = len(class_names)
num_images_per_class = 4  # You can adjust this as per your preference
fig, axes = plt.subplots(num_classes, num_images_per_class, figsize=(12, 12))

for i, class_name in enumerate(class_names):
    # Get indices of images for the current class
    class_indices = np.where(y_test == class_name)[0]

    # Randomly select num_images_per_class correct and incorrect predictions
    np.random.shuffle(class_indices)
    selected_correct_indices = class_indices[y_test[class_indices] == y_pred[class_indices]][:num_images_per_class // 2]
    selected_incorrect_indices = class_indices[y_test[class_indices] != y_pred[class_indices]][:num_images_per_class // 2]

    # Plot correct predictions
    for j, correct_idx in enumerate(selected_correct_indices):
        ax = axes[i, j * 2]
        ax.imshow(X_test[correct_idx], cmap='gray')
        ax.axis('off')
        ax.set_title(f"Correct: {class_name}")

    # Plot incorrect predictions
    for j, incorrect_idx in enumerate(selected_incorrect_indices):
        ax = axes[i, j * 2 + 1]
        ax.imshow(X_test[incorrect_idx], cmap='gray')
        ax.axis('off')
        predicted_label = y_pred[incorrect_idx]
        ax.set_title(f"Predicted: {predicted_label}, Actual: {class_name}")

# Adjust spacing between subplots
plt.tight_layout()
plt.show()

In [ ]:
# Initialize the individual SVM classifiers with different subsets of the training data
svm_classifier_1 = SVC(kernel='linear', C=10, gamma = 'scale', probability=True)
svm_classifier_2 = SVC(kernel='rbf', C=10, probability=True)

# Create the ensemble of SVM classifiers using VotingClassifier
ensemble_classifier = VotingClassifier(estimators=[('linear', svm_classifier_1), ('rbf', svm_classifier_2)], voting='soft')

# Train the ensemble on the full training data
ensemble_classifier.fit(X_train_flat, y_train)

# Predict labels for the test set using the ensemble
y_pred_test = ensemble_classifier.predict(X_test_flat)

# Calculate accuracy on the test set
accuracy_test = accuracy_score(y_test, y_pred_test)
print("Ensemble Testing Accuracy:", accuracy_test)

# Predict labels for the training set using the ensemble
y_pred_train = ensemble_classifier.predict(X_train_flat)

# Calculate accuracy on the training set
accuracy_train = accuracy_score(y_train, y_pred_train)
print("Ensemble Training Accuracy:", accuracy_train)

In [ ]:
# Generate and print the classification report
classification_rep = classification_report(y_test, y_pred, target_names=class_names)
print("Classification Report:")
print(classification_rep)

In [ ]:
svm_res2 = GetModelResultsInDataFrame(results("Ensemble-SVM", y_pred, y_test, y_train, y_pred_train))
svm_res2

In [ ]:
# Initialize parameters for bootstrapping
num_bootstrap_samples = 10  # Number of bootstrap samples
ensemble_accuracies = []

# Perform bootstrapping
for _ in range(num_bootstrap_samples):
    # Create a bootstrap sample
    bootstrap_indices = np.random.choice(len(X_test_flat), len(X_test_flat), replace=True)
    bootstrap_X = X_test_flat[bootstrap_indices]
    bootstrap_y = y_test[bootstrap_indices]

    # Predict class labels using the bagging ensemble
    bootstrap_y_pred = ensemble_classifier.predict(bootstrap_X)

    # Calculate accuracy for the bootstrap sample
    bootstrap_accuracy = accuracy_score(bootstrap_y, bootstrap_y_pred)
    ensemble_accuracies.append(bootstrap_accuracy)

# Calculate mean and confidence interval for ensemble accuracy
mean_accuracy = np.mean(ensemble_accuracies)
lower_bound = np.percentile(ensemble_accuracies, 2.5)
upper_bound = np.percentile(ensemble_accuracies, 97.5)

# Print uncertainty quantification results
print(f"Ensemble Mean Accuracy: {mean_accuracy:.4f}")
print(f"Confidence Interval: {lower_bound:.4f} - {upper_bound:.4f}")

In [ ]:
y_prob = ensemble_classifier.predict_proba(X_test_flat)  # Probability estimates for each class

# Assuming you want to visualize uncertainty by plotting probability distributions
plt.figure(figsize=(12, 6))
for i, class_name in enumerate(class_names):
    class_probs = y_prob[:, i]
    plt.subplot(2, 5, i + 1)
    plt.hist(class_probs, bins=20, alpha=0.7, color='blue')
    plt.title(f'Class {class_name} Probabilities')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots to display images
num_classes = len(class_names)
num_images_per_class = 4  # You can adjust this as per your preference
fig, axes = plt.subplots(num_classes, num_images_per_class, figsize=(12, 12))

for i, class_name in enumerate(class_names):
    # Get indices of images for the current class
    class_indices = np.where(y_test == class_name)[0]

    # Randomly select num_images_per_class correct and incorrect predictions
    np.random.shuffle(class_indices)
    selected_correct_indices = class_indices[y_test[class_indices] == y_pred[class_indices]][:num_images_per_class // 2]
    selected_incorrect_indices = class_indices[y_test[class_indices] != y_pred[class_indices]][:num_images_per_class // 2]

    # Plot correct predictions
    for j, correct_idx in enumerate(selected_correct_indices):
        ax = axes[i, j * 2]
        ax.imshow(X_test[correct_idx], cmap='gray')
        ax.axis('off')
        ax.set_title(f"Correct: {class_name}")

    # Plot incorrect predictions
    for j, incorrect_idx in enumerate(selected_incorrect_indices):
        ax = axes[i, j * 2 + 1]
        ax.imshow(X_test[incorrect_idx], cmap='gray')
        ax.axis('off')
        predicted_label = y_pred[incorrect_idx]
        ax.set_title(f"Predicted: {predicted_label}, Actual: {class_name}")

# Adjust spacing between subplots
plt.tight_layout()
plt.show()

In [ ]:
pd.concat([svm_res, svm_res1, svm_res2])